# Análisis de Patrones de Uso y Modelado Predictivo: Caso Cyclistic

## Descripción del Proyecto
Este proyecto de Ciencia de Datos analiza los patrones de comportamiento de los usuarios del servicio de bicicletas compartidas Cyclistic (basado en datos reales de Divvy Bikes, Chicago). El objetivo estratégico es proporcionar inteligencia de negocio accionable para maximizar la tasa de conversión de usuarios ocasionales (Casual) a miembros de suscripción anual (Member).

El análisis procesa más de 6 millones de registros históricos, abarcando un ciclo anual completo (septiembre 2025 - agosto 2026), e integra técnicas de minería de datos, análisis geoespacial y modelado predictivo mediante Machine Learning.

## Objetivos
1. **Análisis Exploratorio (EDA):** Cuantificar las diferencias en los patrones de uso temporal (duración, demanda semanal y estacionalidad) entre los distintos segmentos de usuarios.
2. **Topología Geoespacial:** Identificar y mapear las estaciones de mayor demanda para focalizar esfuerzos logísticos y de marketing.
3. **Modelado Predictivo:** Entrenar un algoritmo de clasificación interpretable para predecir el tipo de usuario basado en la telemetría del viaje y extraer la importancia paramétrica de las variables.

## Tecnologías y Librerías Utilizadas
* **Lenguaje:** Python 3
* **Manipulación de Datos:** pandas
* **Visualización:** matplotlib, seaborn, folium (mapas interactivos)
* **Machine Learning:** scikit-learn (DecisionTreeClassifier, LabelEncoder, undersampling)
* **Entorno de Desarrollo:** Kaggle Notebooks

## Estructura y Metodología
El proyecto se divide en las siguientes fases metodológicas:
* **Fases 1-4 (Ingeniería de Datos):** Ingesta dinámica de múltiples CSVs (glob), consolidación del dataset (>6M filas), limpieza de valores nulos, conversión de tipos de datos temporales (datetime) y feature engineering.
* **Fases 5-7 (Análisis Descriptivo):** Evaluación de métricas volumétricas y temporales. Generación de cartografía interactiva para el estudio de clústeres de demanda.
* **Fases 8-10 (Machine Learning):** Entrenamiento de un Árbol de Decisión. Identificación de sesgo por clases desbalanceadas. Optimización del modelo mediante técnicas de submuestreo (undersampling) y extracción del Feature Importance.

## Hallazgos Clave
* **Dos Arquetipos Inconfundibles:** Se identificó al "Commuter" (Miembro Anual, trayectos cortos, días laborables, zonas financieras) frente al "Explorador" (Usuario Ocasional, trayectos largos, fines de semana, zonas costeras y turísticas).
* **Validación Predictiva:** El modelo de Machine Learning confirmó que la **duración del viaje** (47.6%) y el **día de la semana** (32.7%) son los predictores absolutos del comportamiento del usuario.

## Recomendaciones Estratégicas
Basado en los datos predictivos y descriptivos, se recomienda a la gerencia:
1. Creación de un "Pase de Fin de Semana" (Flex Pass) para captar al usuario turístico.
2. Lanzamiento de campañas de marketing hiper-localizadas mediante geofencing en las 10 estaciones de mayor demanda ocasional durante los fines de semana de primavera y verano.
3. Optimización logística proactiva de la flota de bicicletas basada en los flujos descubiertos.

## tags
data-cleaning, eda, data-visualization, machine-learning, classification, decision-tree

## Autor
**Enrique Medina Galán**
*Especialización en Inteligencia Artificial y Data Science*

In [ ]:
# --- FASE 1: CARGA Y COMBINACIÓN DE DATOS ---

# Se importa la librería pandas, fundamental para la manipulación y análisis de datos en Python.
import pandas as pd

# Se importa la librería glob, que permite encontrar ficheros que siguen un patrón específico.
import glob

# Se define la ruta donde se encuentran los ficheros CSV cargados en el entorno de Kaggle.
# El patrón '/kaggle/input/datos-cyclistic-2025-2026/*.csv' busca todos los ficheros que terminen en .csv
# dentro del directorio especificado.
ruta_ficheros = '/kaggle/input/datasets/enriquemedinagalan/datos-entrada/*.csv'


# Se utiliza glob.glob para obtener una lista con las rutas completas de todos los ficheros CSV.
lista_rutas_csv = glob.glob(ruta_ficheros)

# Se muestra la lista de ficheros encontrados para verificación.
print(f"Ficheros encontrados: {len(lista_rutas_csv)}")
print(lista_rutas_csv[:5]) # Muestra los primeros 5 para no saturar la salida

# Se inicializa una lista vacía que servirá para almacenar los DataFrames de cada mes.
lista_dataframes = []

# Se itera sobre cada una de las rutas de los ficheros CSV encontrados.
for fichero in lista_rutas_csv:
    # Se lee el fichero CSV y se convierte en un DataFrame de pandas.
    df_mensual = pd.read_csv(fichero)
    # Se añade el DataFrame recién creado a la lista de DataFrames.
    lista_dataframes.append(df_mensual)
    print(f"Fichero '{fichero}' cargado exitosamente.")

# Se utiliza la función pd.concat para combinar todos los DataFrames de la lista en uno solo.
# 'ignore_index=True' asegura que el índice del nuevo DataFrame sea continuo y no repetido.
viajes_df = pd.concat(lista_dataframes, ignore_index=True)


# --- FASE 2: INSPECCIÓN INICIAL DEL DATAFRAME COMBINADO ---

# Se muestra el número total de filas y columnas del DataFrame combinado.
# Esto da una idea inicial del volumen de datos con el que se va a trabajar.
print("\nDimensiones del DataFrame combinado (filas, columnas):")
print(viajes_df.shape)

# Se muestra un resumen conciso del DataFrame.
# .info() proporciona información vital: número de entradas, nombre de cada columna,
# cantidad de valores no nulos y el tipo de dato (Dtype) de cada columna.
# Este paso es crucial para detectar problemas iniciales como tipos de datos incorrectos o valores nulos.
print("\nInformación general del DataFrame:")
viajes_df.info()

# Se muestra una vista previa de las primeras 5 filas del DataFrame.
# .head() permite verificar que los datos se han cargado y combinado correctamente.
print("\nVista previa de los datos:")
viajes_df.head()

In [ ]:
# --- FASE 3: LIMPIEZA Y TRANSFORMACIÓN DE DATOS ---

# Se crea una copia del DataFrame original para realizar las operaciones de limpieza.
# Es una buena práctica en ciencia de datos para no modificar los datos brutos cargados inicialmente.
viajes_limpios_df = viajes_df.copy()

# 1. CONVERSIÓN DE TIPOS DE DATOS DE FECHA
# Se convierten las columnas 'started_at' y 'ended_at' de tipo 'object' a 'datetime'.
# esencial para poder realizar calculos y extracciones basadas en el tiempo.
# El formato se infiere automáticamente, pero se podría especificar con el argumento format='%Y-%m-%d %H:%M:%S'.
viajes_limpios_df['started_at'] = pd.to_datetime(viajes_limpios_df['started_at'])
viajes_limpios_df['ended_at'] = pd.to_datetime(viajes_limpios_df['ended_at'])

# 2. CÁLCULO DE LA DURACIÓN DEL VIAJE
# Se crea una nueva columna llamada 'ride_length' que almacena la duración de cada viaje.
# Se calcula restando la fecha de inicio ('started_at') de la fecha de fin ('ended_at').
# El resultado es un objeto Timedelta, que representa una duración.
viajes_limpios_df['ride_length'] = viajes_limpios_df['ended_at'] - viajes_limpios_df['started_at']

# 3. EXTRACCIÓN DE COMPONENTES DE FECHA
# Se crea una nueva columna 'day_of_week' para almacenar el día de la semana del inicio del viaje.
# El atributo .dt.day_name() extrae el nombre del día (e.g., 'Monday', 'Tuesday').
viajes_limpios_df['day_of_week'] = viajes_limpios_df['started_at'].dt.day_name()

# Se crea la columna 'month' para el mes del inicio del viaje.
viajes_limpios_df['month'] = viajes_limpios_df['started_at'].dt.month_name()

# Se crea la columna 'hour' para la hora del día en que inició el viaje.
viajes_limpios_df['hour'] = viajes_limpios_df['started_at'].dt.hour


# --- FASE 4: FILTRADO DE DATOS INVÁLIDOS ---

# Se realiza una inspección inicial de la nueva columna 'ride_length'.
# La función .describe() proporciona estadísticas descriptivas (media, desviación, mínimos, máximos).
# Esto es útil para detectar valores anómalos, como duraciones negativas.
print("Estadísticas descriptivas de 'ride_length' ANTES del filtrado:")
print(viajes_limpios_df['ride_length'].describe())

# Se procede a eliminar registros que no son lógicos para el análisis.
# Condición 1: Viajes con duración negativa. Esto ocurre si la bicicleta se devuelve en una estación
# con un reloj desincronizado o por algún error en el sistema.
# Condición 2: Viajes con duración inferior a 60 segundos. Se consideran viajes "falsos" o errores,
# donde un usuario podría haber desbloqueado y bloqueado la bicicleta casi inmediatamente.
# Estos registros atípicos ('outliers') pueden distorsionar las métricas agregadas como la duración media.
viajes_limpios_df = viajes_limpios_df[viajes_limpios_df['ride_length'] >= pd.to_timedelta('60s')]

# Se realiza una nueva inspección de 'ride_length' para verificar que el filtrado fue exitoso.
print("\nEstadísticas descriptivas de 'ride_length' DESPUÉS del filtrado:")
print(viajes_limpios_df['ride_length'].describe())

# Se muestra una vista previa del DataFrame transformado, con las nuevas columnas.
print("\nVista previa del DataFrame con las nuevas columnas y datos limpios:")
viajes_limpios_df.head()

In [ ]:
# --- FASE 5: ANÁLISIS EXPLORATORIO DE DATOS (EDA) ---

# Se importan las librerías de visualización: Matplotlib y Seaborn.
# Matplotlib es la librería base para crear gráficos.
# Seaborn se construye sobre Matplotlib y permite crear gráficos estadísticos más atractivos y complejos con menos código.
import matplotlib.pyplot as plt
import seaborn as sns

# Se configura el estilo de los gráficos para que sean visualmente más agradables.
sns.set_style('whitegrid')

# 1. ANÁLISIS GENERAL: NÚMERO TOTAL DE VIAJES Y DURACIÓN MEDIA POR TIPO DE USUARIO

# Se calcula el número total de viajes para cada tipo de usuario.
print("Número total de viajes por tipo de usuario:")
print(viajes_limpios_df['member_casual'].value_counts())

# Se calcula la duración media de los viajes para cada tipo de usuario.
# El resultado es en formato Timedelta, que puede ser difícil de interpretar.
print("\nDuración media del viaje por tipo de usuario (formato Timedelta):")
print(viajes_limpios_df.groupby('member_casual')['ride_length'].mean())

# Para una mejor interpretación, se convierte la duración media a minutos totales.
# Se accede al total de segundos (.dt.total_seconds()) y se divide por 60.
duracion_media_minutos = (viajes_limpios_df.groupby('member_casual')['ride_length'].mean().dt.total_seconds() / 60)
print("\nDuración media del viaje por tipo de usuario (en minutos):")
print(duracion_media_minutos)


# 2. VISUALIZACIÓN 1: NÚMERO DE VIAJES POR DÍA DE LA SEMANA

# Se crea la figura y los ejes para el gráfico. Un tamaño de (12, 6) es adecuado para una buena visualización.
plt.figure(figsize=(12, 6))

# Se utiliza seaborn.countplot para crear un gráfico de barras que cuenta las ocurrencias de cada categoría.
# 'x' es el eje de las categorías (días de la semana).
# 'hue' permite diferenciar las barras por una segunda categoría (tipo de usuario).
# 'order' se usa para asegurar que los días de la semana se muestren en el orden correcto (Lunes a Domingo) y no alfabéticamente.
dias_ordenados = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.countplot(x='day_of_week', hue='member_casual', data=viajes_limpios_df, order=dias_ordenados)

# Se añaden títulos y etiquetas para que el gráfico sea autoexplicativo.
plt.title('Número de Viajes por Día de la Semana', fontsize=16)
plt.xlabel('Día de la Semana', fontsize=12)
plt.ylabel('Total de Viajes', fontsize=12)
plt.legend(title='Tipo de Usuario')

# Se muestra el gráfico.
plt.show()


# 3. VISUALIZACIÓN 2: DURACIÓN MEDIA DE VIAJES POR DÍA DE LA SEMANA

# Se agrupan los datos por tipo de usuario y día de la semana, calculando la duración media en minutos.
# Se resetea el índice (.reset_index()) para convertir el resultado agrupado de nuevo en un DataFrame.
duracion_media_dia = viajes_limpios_df.groupby(['member_casual', 'day_of_week'])['ride_length'].mean().dt.total_seconds() / 60
duracion_media_dia_df = duracion_media_dia.reset_index()

# Se crea la figura y los ejes para el segundo gráfico.
plt.figure(figsize=(12, 6))

# Se utiliza seaborn.barplot para comparar la duración media.
# A diferencia de countplot, barplot muestra la media de una variable numérica ('y') para cada categoría ('x').
sns.barplot(x='day_of_week', y='ride_length', hue='member_casual', data=duracion_media_dia_df, order=dias_ordenados)

# Se añaden títulos y etiquetas.
plt.title('Duración Media de Viaje por Día de la Semana', fontsize=16)
plt.xlabel('Día de la Semana', fontsize=12)
plt.ylabel('Duración Media del Viaje (Minutos)', fontsize=12)
plt.legend(title='Tipo de Usuario')

# Se muestra el gráfico.
plt.show()

In [ ]:
# --- FASE 6: ANÁLISIS DE ESTACIONALIDAD Y PATRONES DIARIOS ---

# Se importan de nuevo las librerías por si esta celda se ejecuta de forma independiente.
import matplotlib.pyplot as plt
import seaborn as sns

# Se configura el estilo de los gráficos.
sns.set_style('whitegrid')


# 1. VISUALIZACIÓN 3: NÚMERO DE VIAJES POR MES

# Se crea la figura y los ejes para el gráfico.
plt.figure(figsize=(14, 7))

# Se define el orden correcto de los meses para una visualización cronológica.
# La columna 'month' se creó en el paso de limpieza.
meses_ordenados = [
    'January', 'February', 'March', 'April', 'May', 'June', 
    'July', 'August', 'September', 'October', 'November', 'December'
]

# Se filtran los meses para que solo se incluyan los presentes en el DataFrame,
# manteniendo el orden cronológico. Esto evita errores si un mes no tuviera datos.
meses_presentes = [mes for mes in meses_ordenados if mes in viajes_limpios_df['month'].unique()]

# Se utiliza seaborn.countplot para contar los viajes por mes.
sns.countplot(x='month', hue='member_casual', data=viajes_limpios_df, order=meses_presentes)

# Se añaden títulos y etiquetas.
plt.title('Distribución Mensual de Viajes por Tipo de Usuario', fontsize=16)
plt.xlabel('Mes', fontsize=12)
plt.ylabel('Total de Viajes', fontsize=12)
# Se rotan las etiquetas del eje X para mejorar la legibilidad.
plt.xticks(rotation=45)
plt.legend(title='Tipo de Usuario')

# Se muestra el gráfico.
plt.show()


# 2. VISUALIZACIÓN 4: NÚMERO DE VIAJES POR HORA DEL DÍA

# Se crea la figura y los ejes para el cuarto gráfico.
plt.figure(figsize=(14, 7))

# Se utiliza seaborn.countplot para contar los viajes por hora del día.
# La columna 'hour' se creó en el paso de limpieza.
sns.countplot(x='hour', hue='member_casual', data=viajes_limpios_df)

# Se añaden títulos y etiquetas.
plt.title('Distribución de Viajes por Hora del Día', fontsize=16)
plt.xlabel('Hora del Día (Formato 24h)', fontsize=12)
plt.ylabel('Total de Viajes', fontsize=12)
plt.legend(title='Tipo de Usuario')

# Se muestra el gráfico.
plt.show()

In [ ]:
# --- FASE 7: ANÁLISIS GEOESPACIAL ---

# Se importa la librería Folium para la creación de mapas interactivos.
import folium

# 1. PREPARACIÓN DE LOS DATOS PARA EL ANÁLISIS GEOESPACIAL

# El análisis de estaciones requiere que los nombres de las estaciones no sean nulos.
# Se crea un DataFrame específico para este análisis eliminando las filas donde 'start_station_name' es nulo.
# Esto asegura que cada viaje contado esté asociado a una estación conocida.
viajes_geo_df = viajes_limpios_df.dropna(subset=['start_station_name', 'start_lat', 'start_lng'])

# 2. IDENTIFICACIÓN DE LAS ESTACIONES MÁS POPULARES

# Se agrupan los datos por tipo de usuario y nombre de la estación de inicio, y se cuenta el número de viajes.
# .size() cuenta el número de filas en cada grupo y .reset_index(name='trip_count') lo convierte en un DataFrame.
conteo_estaciones = viajes_geo_df.groupby(['member_casual', 'start_station_name']).size().reset_index(name='trip_count')

# Se identifican las 10 estaciones más populares para los usuarios 'casual'.
# Se filtra por 'member_casual' == 'casual' y se ordenan los resultados por 'trip_count' de forma descendente.
top_10_casual = conteo_estaciones[conteo_estaciones['member_casual'] == 'casual'].nlargest(10, 'trip_count')

# Se repite el proceso para los usuarios 'member'.
top_10_member = conteo_estaciones[conteo_estaciones['member_casual'] == 'member'].nlargest(10, 'trip_count')

# Se muestran los resultados en formato de tabla para su posterior uso en el informe.
print("--- Top 10 Estaciones de Inicio para Usuarios Ocasionales (Casual) ---")
print(top_10_casual)
print("\n--- Top 10 Estaciones de Inicio para Miembros Anuales (Member) ---")
print(top_10_member)

# 3. CREACIÓN DEL MAPA INTERACTIVO

# Se obtiene un conjunto de datos único con las coordenadas de cada estación para evitar duplicados.
# Se agrupa por nombre de estación y se toma la media de la latitud y longitud.
# Esto soluciona pequeñas variaciones en las coordenadas registradas para una misma estación.
coordenadas_estaciones = viajes_geo_df.groupby('start_station_name')[['start_lat', 'start_lng']].mean().reset_index()

# Se unen los datos de los top 10 con sus respectivas coordenadas.
top_10_casual = pd.merge(top_10_casual, coordenadas_estaciones, on='start_station_name')
top_10_member = pd.merge(top_10_member, coordenadas_estaciones, on='start_station_name')

# Se inicializa el mapa, centrado en las coordenadas aproximadas de Chicago.
# El 'zoom_start' define el nivel de acercamiento inicial.
mapa_chicago = folium.Map(location=[41.8781, -87.6298], zoom_start=12)

# Se añaden marcadores para las estaciones más populares de los usuarios 'casual'.
for index, row in top_10_casual.iterrows():
    folium.Marker(
        location=[row['start_lat'], row['start_lng']],
        popup=f"<strong>{row['start_station_name']}</strong><br>Viajes Casual: {row['trip_count']}",
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(mapa_chicago)

# Se añaden marcadores para las estaciones más populares de los usuarios 'member'.
for index, row in top_10_member.iterrows():
    folium.Marker(
        location=[row['start_lat'], row['start_lng']],
        popup=f"<strong>{row['start_station_name']}</strong><br>Viajes Member: {row['trip_count']}",
        icon=folium.Icon(color='red', icon='user')
    ).add_to(mapa_chicago)

# Se muestra el mapa en la salida del notebook.
mapa_chicago

In [ ]:
# --- FASE 8: MODELO PREDICTIVO DE CLASIFICACIÓN (MACHINE LEARNING) ---
#Este bloque de código es el más complejo,ciclo de vida del modelado: 
#preparación de datos, entrenamiento, predicción y evaluación.

# Se importan las librerías necesarias de Scikit-Learn.
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# 1. PREPARACIÓN DE DATOS (FEATURE ENGINEERING)

# Se seleccionan las características (variables predictoras) y la variable objetivo.
features = ['ride_length', 'day_of_week', 'hour']
target = 'member_casual'

# Se crea un nuevo DataFrame para el modelo, eliminando valores nulos en las columnas seleccionadas.
# Aunque ya se limpiaron, es una buena práctica para asegurar la integridad.
df_modelo = viajes_limpios_df[features + [target]].dropna()

# Se convierte 'ride_length' de Timedelta a segundos totales (un número) para que el modelo pueda procesarlo.
df_modelo['ride_length_sec'] = df_modelo['ride_length'].dt.total_seconds()

# Se utiliza LabelEncoder para convertir las variables categóricas ('day_of_week' y 'member_casual') a números.
# El modelo de ML solo puede trabajar con datos numéricos.
le_day = LabelEncoder()
le_target = LabelEncoder()

df_modelo['day_of_week_encoded'] = le_day.fit_transform(df_modelo['day_of_week'])
df_modelo['target_encoded'] = le_target.fit_transform(df_modelo[target])

# Se definen las variables finales para el modelo.
X = df_modelo[['ride_length_sec', 'day_of_week_encoded', 'hour']]
y = df_modelo['target_encoded']

# Debido al gran volumen de datos (más de 5 millones de filas), entrenar con todo el dataset
# puede ser muy lento y consumir mucha memoria. Se tomará una muestra aleatoria representativa.
# 500,000 registros son más que suficientes para entrenar un modelo robusto.
df_sample = df_modelo.sample(n=500000, random_state=42)
X_sample = df_sample[['ride_length_sec', 'day_of_week_encoded', 'hour']]
y_sample = df_sample['target_encoded']

# 2. DIVISIÓN DE DATOS EN ENTRENAMIENTO Y PRUEBA

# Se dividen los datos de la muestra en un conjunto para entrenar el modelo (80%) y otro para evaluarlo (20%).
# 'random_state=42' asegura que la división sea siempre la misma, haciendo el resultado reproducible.
X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2, random_state=42)

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape[0]} registros")
print(f"Tamaño del conjunto de prueba: {X_test.shape[0]} registros")

# 3. ENTRENAMIENTO DEL MODELO DE ÁRBOL DE DECISIÓN

# Se inicializa el clasificador de Árbol de Decisión.
# 'max_depth=5' es un hiperparámetro crucial para prevenir el sobreajuste.
# Limita la profundidad del árbol a 5 niveles de decisiones, forzándolo a aprender solo los patrones más generales.
modelo_arbol = DecisionTreeClassifier(max_depth=5, random_state=42)

# Se entrena el modelo utilizando los datos de entrenamiento.
modelo_arbol.fit(X_train, y_train)

# 4. EVALUACIÓN DEL MODELO

# Se utiliza el modelo entrenado para hacer predicciones sobre el conjunto de prueba (datos que nunca ha visto).
y_pred = modelo_arbol.predict(X_test)

# Se calcula la precisión (accuracy) del modelo.
accuracy = accuracy_score(y_test, y_pred)
print(f"\nPrecisión (Accuracy) del modelo: {accuracy:.4f}")

# Se genera un informe de clasificación completo, con métricas como precisión, recall y f1-score por cada clase.
print("\nInforme de Clasificación:")
# Se usan las clases originales ('casual', 'member') para un informe más legible.
target_names = le_target.inverse_transform(modelo_arbol.classes_)
print(classification_report(y_test, y_pred, target_names=target_names))

# Se crea y visualiza una matriz de confusión para entender los tipos de errores del modelo.
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Matriz de Confusión')
plt.ylabel('Etiqueta Real')
plt.xlabel('Etiqueta Predicha')
plt.show()

In [ ]:
# --- FASE 9: OPTIMIZACIÓN DEL MODELO - MANEJO DE CLASES DESBALANCEADAS ---

# Se importan las librerías necesarias.
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

print("--- Iniciando Experimento de Optimización del Modelo ---")

# 1. PREPARACIÓN DE UN DATASET DE ENTRENAMIENTO BALANCEADO (TÉCNICA DE SUBMUESTREO)

# Se parte del conjunto de entrenamiento original (X_train, y_train) que estaba desbalanceado.
# Se combinan de nuevo para poder filtrar por la clase.
df_train_original = pd.concat([X_train, y_train.rename('target_encoded')], axis=1)

# Se separan las dos clases: la mayoritaria (member) y la minoritaria (casual).
# En nuestro caso, la clase 'casual' (codificada como 0) es la minoritaria.
clase_minoritaria = df_train_original[df_train_original['target_encoded'] == 0]
clase_mayoritaria = df_train_original[df_train_original['target_encoded'] == 1]

# Se realiza el submuestreo (undersampling).
# Se toma una muestra aleatoria de la clase mayoritaria que tenga el mismo tamaño que la clase minoritaria.
clase_mayoritaria_submuestreada = clase_mayoritaria.sample(n=len(clase_minoritaria), random_state=42)

# Se combinan de nuevo las dos clases para crear el nuevo dataset de entrenamiento balanceado.
df_train_balanceado = pd.concat([clase_minoritaria, clase_mayoritaria_submuestreada])

# Se verifica que el nuevo dataset de entrenamiento está perfectamente balanceado.
print("\nDistribución de clases en el nuevo conjunto de entrenamiento balanceado:")
print(df_train_balanceado['target_encoded'].value_counts())

# Se preparan las variables X e y para el nuevo modelo.
X_train_balanceado = df_train_balanceado.drop('target_encoded', axis=1)
y_train_balanceado = df_train_balanceado['target_encoded']


# 2. ENTRENAMIENTO DEL NUEVO MODELO CON DATOS BALANCEADOS

# Se inicializa un nuevo clasificador de Árbol de Decisión.
modelo_arbol_balanceado = DecisionTreeClassifier(max_depth=5, random_state=42)

# Se entrena el nuevo modelo utilizando los datos de entrenamiento BALANCEADOS.
modelo_arbol_balanceado.fit(X_train_balanceado, y_train_balanceado)

# 3. EVALUACIÓN DEL NUEVO MODELO (SOBRE EL MISMO CONJUNTO DE PRUEBA)

# Es crucial evaluar el nuevo modelo sobre el mismo conjunto de prueba (X_test, y_test) original,
# que sigue teniendo la distribución de clases del mundo real.
y_pred_balanceado = modelo_arbol_balanceado.predict(X_test)

# Se calcula la precisión (accuracy) del nuevo modelo.
accuracy_balanceado = accuracy_score(y_test, y_pred_balanceado)
print(f"\nPrecisión (Accuracy) del modelo BALANCEADO: {accuracy_balanceado:.4f}")

# Se genera un informe de clasificación completo para el nuevo modelo.
print("\nInforme de Clasificación del modelo BALANCEADO:")
print(classification_report(y_test, y_pred_balanceado, target_names=target_names))

# Se crea y visualiza la matriz de confusión del nuevo modelo.
cm_balanceado = confusion_matrix(y_test, y_pred_balanceado)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_balanceado, annot=True, fmt='d', cmap='Greens', xticklabels=target_names, yticklabels=target_names)
plt.title('Matriz de Confusión (Modelo Balanceado)')
plt.ylabel('Etiqueta Real')
plt.xlabel('Etiqueta Predicha')
plt.show()

In [ ]:
# --- FASE 10: INTERPRETACIÓN DEL MODELO Y ANÁLISIS DE IMPORTANCIA DE CARACTERÍSTICAS ---

# Se importan las librerías necesarias.
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
import pandas as pd

print("--- Análisis de Importancia de Características del Modelo Balanceado ---")

# 1. VISUALIZACIÓN DEL ÁRBOL DE DECISIÓN

# El modelo 'modelo_arbol_balanceado' ya está entrenado.
# Ahora se visualizarán las reglas que ha aprendido.
plt.figure(figsize=(25, 15))

# Se utiliza la función plot_tree para dibujar el árbol.
# 'feature_names' son los nombres de nuestras columnas predictoras.
# 'class_names' son los nombres de nuestras clases objetivo ('casual', 'member').
# 'filled=True' colorea los nodos según la clase mayoritaria.
# 'rounded=True' usa esquinas redondeadas en los nodos.
# 'fontsize' ajusta el tamaño del texto para mayor legibilidad.
plot_tree(modelo_arbol_balanceado, 
          feature_names=X_train_balanceado.columns, 
          class_names=target_names, 
          filled=True, 
          rounded=True, 
          fontsize=10)

# Se muestra el gráfico del árbol.
plt.title("Visualización del Árbol de Decisión Entrenado (Modelo Balanceado)", fontsize=20)
plt.show()


# 2. ANÁLISIS DE LA IMPORTANCIA DE LAS CARACTERÍSTICAS

# Un modelo de árbol de decisión calcula una puntuación de "importancia" para cada característica.
# Esta puntuación indica cuánto contribuye cada variable a la reducción de la impureza (mejora de la predicción).
importancias = modelo_arbol_balanceado.feature_importances_

# Se crea un DataFrame de pandas para visualizar mejor estas puntuaciones.
df_importancias = pd.DataFrame({
    'Caracteristica': X_train_balanceado.columns,
    'Importancia': importancias
}).sort_values('Importancia', ascending=False)

print("\nImportancia de cada característica para el modelo:")
print(df_importancias)


# Se crea un gráfico de barras para visualizar la importancia de las características.
plt.figure(figsize=(10, 6))
sns.barplot(x='Importancia', y='Caracteristica', data=df_importancias, palette='viridis')
plt.title('Importancia de las Características en el Modelo Predictivo', fontsize=16)
plt.xlabel('Puntuación de Importancia', fontsize=12)
plt.ylabel('Característica', fontsize=12)
plt.show()

In [ ]:
%%writefile requirements.txt
pandas>=2.0.0
matplotlib>=3.7.0
seaborn>=0.12.0
folium>=0.14.0
scikit-learn>=1.2.0